In [1]:
import os

os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=4"
import jax
import jax.numpy as jnp

jax.devices()

[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3)]

In [2]:
from functools import partial

from jax.experimental.mesh_utils import create_device_mesh
from jax.experimental.multihost_utils import process_allgather
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

all_gather = partial(process_allgather, tiled=False)

pdims = (2, 2)
devices = create_device_mesh(pdims)
mesh = Mesh(devices, axis_names=("x", "y"))
sharding = NamedSharding(mesh, P("x", "y"))

In [31]:
from jaxpm.distributed import normal_field
from numpyro.distributions import Normal, constraints
from numpyro.distributions.util import promote_shapes
from numpyro.util import is_prng_key


class DistributedNormal(Normal):
    arg_constraints = {"loc": constraints.real, "scale": constraints.positive}
    support = constraints.real
    reparametrized_params = ["loc", "scale"]

    def __init__(self, loc=0.0, scale=1.0, sharding=None, *, validate_args=None):
        self.loc, self.scale = promote_shapes(loc, scale)
        self.sharding = sharding
        batch_shape = jax.lax.broadcast_shapes(jnp.shape(loc), jnp.shape(scale))
        super(Normal, self).__init__(batch_shape=batch_shape, validate_args=validate_args)

    def sample(self, key, sample_shape=()):
        assert is_prng_key(key)
        eps = normal_field(sample_shape + self.batch_shape + self.event_shape, key, self.sharding)
        return self.loc + eps * self.scale


box_shape = (16, 16, 16)

dist = DistributedNormal(jnp.zeros(box_shape), jnp.ones(box_shape), sharding=sharding)

In [33]:
nn = Normal(jnp.zeros(box_shape), jnp.ones(box_shape))
nn.sample(jax.random.PRNGKey(0)).shape

(16, 16, 16)

In [35]:
dist.sample(jax.random.PRNGKey(0)).sharding

NamedSharding(mesh=Mesh('x': 2, 'y': 2), spec=PartitionSpec('x', 'y'), memory_kind=unpinned_host)